#### ***LangSmith***

#### **LangSmith is an enterprise grade AI Engineering Platform built by the team behind LangChain. It is mainly used to trace,debug,evaluate,deploy, and monitor applications powered by LLM.**

In [23]:
from dotenv import load_dotenv

load_dotenv()

True

In [24]:
### Check Configuration

import os
print("Tracing:",os.environ['LANGSMITH_TRACING'])
print("Project:",os.environ['LANGSMITH_PROJECT'])

Tracing: true
Project: production-rag-kubernetes


In [25]:
###create a embedding by using langchain hugging face.

from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
        model = "BAAI/bge-large-en-v1.5",
        model_kwargs = {"device":"cpu"},
        encode_kwargs = {"normalize_embeddings":True}
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [26]:
### Create LLM
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b"
)

llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.16'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000020EFC87C860>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000020EFC87D100>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [27]:
## Load the vector store from Chroma
from langchain_chroma import Chroma
vectorstore = Chroma(
    persist_directory="../vectorstore/kubernetes_rag",
    collection_name = "kubernetes_rag",
    embedding_function=embedding_model
)

In [28]:
### Similarity search
retriever = vectorstore.as_retriever(search_kwargs = {"k":3})

In [29]:
##Design a Prompt
from langchain_core.prompts import ChatPromptTemplate
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


prompt = ChatPromptTemplate.from_template("""
You are a Kubernetes documentation Assistant.

Answer the question using only the provided Context only.


Rules:
    1.Don't use information outside the Context.
    2.If the answer is not available in the context,
    say I don't have information based on the Provided Documents.

Context:
{context}

Question:
{question}

Answer:
""")

In [30]:
from langchain_core.runnables import RunnableLambda,RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
rag_chain = ({
    "context":retriever | RunnableLambda(format_docs),
    "question":RunnablePassthrough()
}
|prompt
| llm 
| StrOutputParser()
)

In [31]:
query = "What is a Kubernetes Deployment?"

In [32]:
response = rag_chain.invoke(query,
    config = {"run_name":"Kubernetes Rag"}
)
print(response)

A **Kubernetes Deployment** is a Kubernetes API object that manages the creation and updating of application instances (Pods) on a cluster. When you create a Deployment you define a **spec** that describes the desired state—such as the number of replicas you want running. The Kubernetes control plane continuously compares the current **status** of the Pods with the spec and automatically makes corrections (e.g., launching replacement Pods if any fail) to ensure the actual state matches the desired state. In short, a Deployment declaratively controls the lifecycle of an application’s Pods, handling scaling, rolling updates, and self‑healing.


In [33]:
test_queries = [
    "What is a Kubernetes Deployment?",
    "What is a Kubernetes Pod?",
    "What is a Kubernetes Service?",
    "What is a ReplicaSet?",
    "What is a ConfigMap?"
]

In [34]:
results = []
for query in test_queries:
    response = rag_chain.invoke(query,
    config = {"run_name":"Kubernetes Rag"})
    results.append({
        "Query":query,
        "Response":response
    })

In [35]:
for result in results:

    print("Question:",result['Query'])
    print("Response:",result['Response'])
    print("#"*70)

Question: What is a Kubernetes Deployment?
Response: A **Kubernetes Deployment** is a Kubernetes API object that manages the creation and updating of application instances (Pods) on a cluster. When you define a Deployment, you provide a **spec** that describes the desired state—such as the number of replica Pods you want running. The Kubernetes control plane continuously compares the current **status** of those Pods with the spec and automatically makes corrections (e.g., launching replacement Pods if any fail) to keep the actual state aligned with the desired state. In short, a Deployment declaratively defines how many copies of an application should run and ensures that the cluster maintains that state over time.
######################################################################
Question: What is a Kubernetes Pod?
Response: A Kubernetes **Pod** is the smallest deployable unit in Kubernetes. It is a logical host that groups one or more containers (such as Docker containers) togeth